In [2]:
# Réinstaller proprement les dépendances
!pip uninstall -y unsloth bitsandbytes
!pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git" --upgrade
!pip install xformers nltk --no-deps

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-624ylb6e/unsloth_9d0a6e26af8243c79f07c505dd33b1a0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-624ylb6e/unsloth_9d0a6e26af8243c79f07c505dd33b1a0
  Resolved https://github.com/unslothai/unsloth.git to commit 5ce83f272bd9e726b68b0a0452b8b04d32477133
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 30.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 26.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 20.3 MB/

In [5]:
import torch
import unsloth
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig
import pandas as pd
from tqdm import tqdm
import os
import regex as re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
from datasets import Dataset
from unsloth import FastLanguageModel

tqdm.pandas()

## Charger le modèle

In [6]:
from huggingface_hub import login

login(login_huggin) 

NameError: name 'login_huggin' is not defined

In [ ]:

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_cache = False,
)

==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.3.
   \\   /|    NVIDIA A100-PCIE-40GB MIG 1g.5gb. Num GPUs = 1. Max memory: 4.75 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


##  Charger et préparer les données (phishing dataset)

In [ ]:

data = pd.read_csv("models/merged_data.csv.zip")  

# Supprimer les colonnes inutiles
data = data.drop(columns=['Unnamed: 0'])


# Gérer les valeurs manquantes
data = data.dropna(subset=['body', 'label'])

data.head(20)


/tmp/ipykernel_4639/1170387343.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("models/merged_data.csv.zip")


,date,body,label
0,"Thu, 31 Oct 2002 02:38:20 +0000",FROM:MR. JAMES NGOLA.\nCONFIDENTIAL TEL: 233-2...,1
1,"Thu, 31 Oct 2002 05:10:00 -0000","Dear Friend,\n\nI am Mr. Ben Suleman a custom ...",1
2,"Thu, 31 Oct 2002 22:17:55 +0100",FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,1
3,"Thu, 31 Oct 2002 22:44:20 -0000",FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,1
4,"Fri, 01 Nov 2002 01:45:04 +0100","Dear sir, \n \nIt is with a heart full of hope...",1
5,"Sat, 02 Nov 2002 06:23:11 +0000",ATTENTION: ...,1
6,NaN,"Dear Sir,\n\nI am Barrister Tunde Dosumu (SAN)...",1
7,"Sun, 03 Nov 2002 23:56:20 +0000",FROM: WILLIAM DRALLO.\nCONFIDENTIAL TEL: 233-2...,1
8,"Mon, 04 Nov 2002 23:41:26 -0000","CHALLENGE SECURITIES LTD.\nLAGOS, NIGERIA\n\n\...",1
9,NaN,"Dear Sir,\n\nI am Barrister Tunde Dosumu (SAN)...",1


## Préparation des données

In [ ]:
import pandas as pd
from multiprocessing import Pool

# Fonction de nettoyage du texte
import re
import string
import unicodedata

def clean_text(text):
    """Nettoie le texte tout en conservant le contenu principal"""
    try:
        # Normalisation des caractères Unicode
        text = unicodedata.normalize("NFKC", str(text))
    except Exception as e:
        print(f"Erreur de normalisation : {e}")
    
    # Conversion en minuscules
    text = text.lower()
    
    # Expressions régulières corrigées avec raw strings
    patterns = [
        r'\[.*?\]',          # Contenu entre crochets
        r'https?://\S+|www\.\S+',  # URLs
        r'<.*?>+',           # Balises HTML
        r'[%s]' % re.escape(string.punctuation.replace('.', '')),  # Ponctuation sauf .
        r'\n',               # Nouvelle ligne
        r'\r',               # Retour chariot
        r'\b\w*\d\w*\b',     # Mots contenant des chiffres
    ]
    
    # Application progressive des nettoyages
    for pattern in patterns:
        text = re.sub(pattern, ' ', text)
    
    # Nettoyage final
    text = re.sub(r'\s+', ' ', text).strip()  # Supprime les espaces multiples
    return text

# Fonction pour appliquer le nettoyage en parallèle
def apply_parallel(df, func, num_workers=4):
    with Pool(num_workers) as pool:
        return pd.Series(pool.map(func, df))

# Appliquer la fonction sur la colonne 'body' avec multiprocessing
data['body'] = apply_parallel(data['body'], clean_text, num_workers=8)

data.head()


,date,body,label
0,"Thu, 31 Oct 2002 02:38:20 +0000",from mr. james ngola. confidential tel . e mai...,1
1,"Thu, 31 Oct 2002 05:10:00 -0000",dear friend i am mr. ben suleman a custom offi...,1
2,"Thu, 31 Oct 2002 22:17:55 +0100",from his royal majesty hrm crown ruler of elem...,1
3,"Thu, 31 Oct 2002 22:44:20 -0000",from his royal majesty hrm crown ruler of elem...,1
4,"Fri, 01 Nov 2002 01:45:04 +0100",dear sir it is with a heart full of hope that ...,1


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Télécharger les stopwords et lemmatizer si nécessaire
nltk.download('stopwords')
nltk.download('wordnet')

# Charger les stopwords et lemmatizer
sw = set(stopwords.words('english') + ['hou', 'ect'])
lemmatizer = WordNetLemmatizer()

# Optimisation de la fonction stop_lem
def stop_lem(text):
    # Séparer les mots et filtrer les stopwords en une seule étape
    words_filtered = [lemmatizer.lemmatize(word) for word in text.split() if word not in sw]
    
    # Joindre les mots lemmatisés en une seule chaîne
    return ' '.join(words_filtered)

# Appliquer sur les données
data['body'] = data['body'].progress_apply(stop_lem)


[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /home/onyxia/nltk_data...
100%|██████████| 164972/164972 [01:11<00:00, 2295.65it/s]


In [ ]:
data.head(20)

,date,body,label
0,"Thu, 31 Oct 2002 02:38:20 +0000",mr. james ngola. confidential tel . e mail jam...,1
1,"Thu, 31 Oct 2002 05:10:00 -0000",dear friend mr. ben suleman custom officer wor...,1
2,"Thu, 31 Oct 2002 22:17:55 +0100",royal majesty hrm crown ruler eleme kingdom ch...,1
3,"Thu, 31 Oct 2002 22:44:20 -0000",royal majesty hrm crown ruler eleme kingdom ch...,1
4,"Fri, 01 Nov 2002 01:45:04 +0100",dear sir heart full hope write seek help respe...,1
5,"Sat, 02 Nov 2002 06:23:11 +0000",attention president managing director dear sir...,1
6,NaN,dear sir barrister tunde dosumu san solicitor ...,1
7,"Sun, 03 Nov 2002 23:56:20 +0000",william drallo. confidential tel . ascetained ...,1
8,"Mon, 04 Nov 2002 23:41:26 -0000",challenge security ltd. lagos nigeria attentio...,1
9,NaN,dear sir barrister tunde dosumu san solicitor ...,1


In [ ]:
data.dropna(subset=['body'], inplace=True)

# Transformer en format conversationnel

def formatting_func(examples):
    return {
        "text": [f"Génère un email de phishing:\n\n{text}\n\n### Réponse:" 
                for text in examples["body"]]
    }

dataset = Dataset.from_pandas(data)
dataset = dataset.map(formatting_func, batched=True)

# Vérification
print(dataset[0])



Map: 100%|██████████| 164972/164972 [00:01<00:00, 135296.40 examples/s]

{'date': 'Thu, 31 Oct 2002 02:38:20 +0000', 'body': 'mr. james ngola. confidential tel . e mail james maktoob.com . urgent business assistance partnership. dear friend dr. james ngola personal assistance late congolese president laurent kabila assassinated body guard jan. . incident occurred presence holding meeting excellency financial return diamond sale area controlled d.r.c. democratic republic congo force foreign ally angola zimbabwe received previous day usd one hundred million united state dollar cash three diplomatic box routed zimbabwe. purpose writing letter solicit assistance cover fund also collaboration moving said fund bank account sum usd twenty five million united state dollar deposited security company ghana diplomatic box gold worth usd twenty five million united state dollar safe keeping security vault investment perhaps country. introduced reliable friend mine traveller also member chamber commerce reliable trustworthy person rely foreign partner even though nature 

## Configurer LoRA pour le fine-tuning 

Pour prendre moins de temps à entrainer

## Lancer le fine-tuning

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    max_seq_length = 2048,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.3.19 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for body in examples["body"]:
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
        Vous êtes un expert en génération d'emails de phishing.<|eot_id|>
        <|start_header_id|>user<|end_header_id|>
        Génère un email de phishing crédible<|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        {body}<|eot_id|>"""
        texts.append(text)
    return {"text" : texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

Map: 100%|██████████| 164972/164972 [00:01<00:00, 129181.86 examples/s]


In [ ]:
split_dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
from unsloth import UnslothTrainingArguments, UnslothTrainer
training_args = UnslothTrainingArguments(
    output_dir='./results',           # Répertoire pour sauvegarder les résultats
    evaluation_strategy="epoch",      # Stratégie d'évaluation
    learning_rate=2e-4,               # Taux d'apprentissage
    per_device_train_batch_size=2,    # Taille du lot d'entraînement
    num_train_epochs=3,               # Nombre d'époques
    weight_decay=0.01,                # Décroissance du poids
    fp16=True,                        # Utiliser float16 pour la précision
    bf16=False,                       # Désactiver bfloat16
)

# Créer l'objet `UnslothTrainer`
trainer = UnslothTrainer(
    model=model,                      # Le modèle à entraîner
    args=training_args,               # Les arguments d'entraînement
    train_dataset=split_dataset["train"],  # Jeu d'entraînement
    eval_dataset=split_dataset["test"],   # Jeu d'évaluation
    tokenizer=tokenizer               # Le tokenizer
)

ModuleNotFoundError: No module named 'unsloth'

In [ ]:
# Exécuter l'entraînement
train_output = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 148,474 | Num Epochs = 3 | Total steps = 222,711
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 3,407,872/1,000,000,000 (0.34% trained)


Epoch,Training Loss,Validation Loss


## Sauvegarder le modèle fine-tuné

In [ ]:

model.save_pretrained("phishing-llama3")
tokenizer.save_pretrained("phishing-llama3")




## Tester la génération

In [ ]:
def generate_phishing_email(prompt):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
    output = model.generate(input_ids, max_length=200)
    return tokenizer.decode(output[0], skip_special_tokens=True)

test_prompt = "Write a phishing email pretending to be PayPal"
print(generate_phishing_email(test_prompt))

In [ ]:





def generate_phishing_email(prompt):
    inputs = tokenizer(
        [prompt],
        return_tensors = "pt",
        padding = True,
    ).to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,
        temperature = 0.7,
        do_sample = True,
    )
    
    return tokenizer.decode(outputs[0])


print(generate_phishing_email("Génère un email de phishing imitant PayPal:"))


In [4]:
import torch
print(torch.cuda.is_available())  # Doit retourner True


print(torch.cuda.get_device_name(0))  # Affiche le nom de ton GPU

True
NVIDIA A100-PCIE-40GB MIG 1g.5gb
